## 4 Movies Crew and Keywords

Once we had clean and filtered the original movies dataset, now we are going to fetch their crew information (such as the actors and directors), as well as their keywords, to have a useful dataset for a content based recommender system.

In [1]:
import ast
import pandas as pd
import numpy as np
import requests
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load the environment variables .env
load_dotenv()

True

Get the id's of the desired films, (the selected ones in the previous notebook)

In [8]:
ids = pd.read_csv('../data/processed/clean_movies_ids.csv')
ids.head()

,movieId,id
0,58559,155
1,109487,157336
2,5618,129
3,7153,122
4,3147,497


Create a function to handle the endpoints for the TMDB api

In [9]:
def fetch_tmdb_data(movie_id, api_key, endpoint=""):
    """Generic function to fetch any TMDB data for a movie"""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/{endpoint}?api_key={api_key}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:  # Too Many Requests
            # Sleep for a bit if we hit rate limits
            time.sleep(2)
            return fetch_tmdb_data(movie_id, api_key, endpoint)  # Simple retry
        else:
            print(f"Error: HTTP {response.status_code} for movie {movie_id}")
            return None
    except Exception as e:
        print(f"Error fetching data for movie {movie_id}: {e}")
        return None

In [10]:
# Create endpoint-specific functions that use the global one
def fetch_credits(movie_id, api_key):
    return fetch_tmdb_data(movie_id, api_key, "credits")

def fetch_keywords(movie_id, api_key):
    return fetch_tmdb_data(movie_id, api_key, "keywords")

In [11]:
def fetch_movies(ids, api_key, fetch_func, max_workers=10, delay=0.05):
    """
    Function to fetch movie data for multiple movie IDs concurrently
    
    Args:
        ids: List of movie IDs
        api_key: TMDB API key
        fetch_func: Function to fetch specific data (e.g., fetch_credits)
        max_workers: Max number of concurrent requests
        delay: Small delay between requests to be nice to API
        
    Returns:
        (successful_results, failed_ids)
    """
    successful = []
    failed = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_func, movie_id, api_key): movie_id for movie_id in ids}
        
        for future in as_completed(futures):
            movie_id = futures[future]
            result = future.result()
            
            if result:
                successful.append(result)
            else:
                failed.append(movie_id)
            
            # Small delay to avoid hammering the API
            time.sleep(delay)
    
    return successful, failed

In [12]:
api_key = os.getenv('tmdb_api_key')
movie_ids = ids['id']

Once we've created the functions to fetch the desired data correctly, we will now begin retrieving the keywords

In [13]:
# Fetching keywords
keywords, failed_keywords = fetch_movies(movie_ids, api_key, fetch_keywords)
print(f"Fetched {len(keywords)} movie keywords, failed {len(failed_keywords)}")

Fetched 5000 movie keywords, failed 0


#### Keywords

In [14]:
keywords = pd.DataFrame(keywords)
keywords.head()

,id,keywords
0,155,"[{'id': 4426, 'name': 'sadism'}, {'id': 4630, ..."
1,157336,"[{'id': 10084, 'name': 'rescue'}, {'id': 2964,..."
2,372058,"[{'id': 6270, 'name': 'high school'}, {'id': 4..."
3,497,"[{'id': 791, 'name': 'mentally disabled'}, {'i..."
4,129,"[{'id': 616, 'name': 'witch'}, {'id': 970, 'na..."


Now retrive the credits information

In [15]:
# Fetching credits
credits, failed_credits = fetch_movies(movie_ids, api_key, fetch_credits)
print(f"Fetched {len(credits)} movie credits, failed {len(failed_credits)}")

Fetched 5000 movie credits, failed 0


#### Credits

In [16]:
credits = pd.DataFrame(credits)
credits.head()

,id,cast,crew
0,121,"[{'adult': False, 'gender': 2, 'id': 109, 'kno...","[{'adult': False, 'gender': 2, 'id': 1319, 'kn..."
1,120,"[{'adult': False, 'gender': 2, 'id': 109, 'kno...","[{'adult': False, 'gender': 2, 'id': 117, 'kno..."
2,496243,"[{'adult': False, 'gender': 2, 'id': 20738, 'k...","[{'adult': False, 'gender': 0, 'id': 3111262, ..."
3,550,"[{'adult': False, 'gender': 2, 'id': 819, 'kno...","[{'adult': False, 'gender': 1, 'id': 7481, 'kn..."
4,497,"[{'adult': False, 'gender': 2, 'id': 31, 'know...","[{'adult': False, 'gender': 2, 'id': 4027, 'kn..."


There was no movie that failed to retrieve its data.

In [17]:
# Ensure we do not have duplicated rows
credits[credits['id'].duplicated()].shape[0], keywords[keywords['id'].duplicated()].shape[0]

(0, 0)

## Extracting the Director and Actors

1. **Crew:** From the crew, we will only extract the director.
2. **Cast:** Lesser known actors and minor roles do not really affect people's opinion of a movie. Therefore, we must only select the major characters and their respective actors, so we will choose the top 3 actors that appear in the credits list. 

In [18]:
# Convertfrom JSON to a Pyhton Object
credits['crew'] = credits['crew'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
credits['cast'] = credits['cast'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

In [19]:
# Extract the director from the crew
def get_director(x):
    for item in x:
        if item['job'] == 'Director':
            return item['name']
    return np.nan

In [20]:
credits['director'] = credits['crew'].apply(lambda x: get_director(x))
# Get the top 3 actors from the cast
credits['cast'] = credits['cast'].apply(lambda x: x[:3] if len(x) >=3 else x).apply(lambda x: [item['name'] for item in x])

In [21]:
credits.head()

,id,cast,crew,director
0,121,"[Elijah Wood, Ian McKellen, Viggo Mortensen]","[{'adult': False, 'gender': 2, 'id': 1319, 'kn...",Peter Jackson
1,120,"[Elijah Wood, Ian McKellen, Viggo Mortensen]","[{'adult': False, 'gender': 2, 'id': 117, 'kno...",Peter Jackson
2,496243,"[Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong]","[{'adult': False, 'gender': 0, 'id': 3111262, ...",Bong Joon Ho
3,550,"[Edward Norton, Brad Pitt, Helena Bonham Carter]","[{'adult': False, 'gender': 1, 'id': 7481, 'kn...",David Fincher
4,497,"[Tom Hanks, David Morse, Bonnie Hunt]","[{'adult': False, 'gender': 2, 'id': 4027, 'kn...",Frank Darabont


Finally we will delete the crew column, since we had already extract the relevant information

In [22]:
# Delete the crew column
credits.drop(columns='crew', inplace=True)

# Nan if the film do not have information about the actors
credits['cast'] = credits['cast'].apply(lambda x: np.nan if not x else x)
credits.isna().sum()

id           0
cast        17
director     1
dtype: int64

We have the complete information for all the movies

## Extracting the Keywords

Now we are going to generate a list of all the keywords for each movie

In [23]:
keywords.iloc[0]['keywords']

[{'id': 4426, 'name': 'sadism'},
 {'id': 4630, 'name': 'chaos'},
 {'id': 1308, 'name': 'secret identity'},
 {'id': 853, 'name': 'crime fighter'},
 {'id': 9715, 'name': 'superhero'},
 {'id': 2095, 'name': 'anti hero'},
 {'id': 3151, 'name': 'scarecrow'},
 {'id': 9717, 'name': 'based on comic'},
 {'id': 7002, 'name': 'vigilante'},
 {'id': 10291, 'name': 'organized crime'},
 {'id': 10044, 'name': 'tragic hero'},
 {'id': 14625, 'name': 'anti villain'},
 {'id': 18023, 'name': 'criminal mastermind'},
 {'id': 33518, 'name': 'district attorney'},
 {'id': 33637, 'name': 'super power'},
 {'id': 163074, 'name': 'super villain'},
 {'id': 207268, 'name': 'neo-noir'},
 {'id': 325778, 'name': 'bold'}]

As we can see the keyword column is a list of dictionaries, corresponding to name and id of each keyword

In [24]:
keywords['keywords'] = keywords['keywords'].apply(lambda x: json.loads(x) if isinstance(x, str) else x) \
                        .apply(lambda x: [item['name'] for item in x]) \
                        .apply(lambda x: np.nan if not x else x)
keywords.head()

,id,keywords
0,155,"[sadism, chaos, secret identity, crime fighter..."
1,157336,"[rescue, future, spacecraft, race against time..."
2,372058,"[high school, race against time, dreams, after..."
3,497,"[mentally disabled, based on novel or book, so..."
4,129,"[witch, parent child relationship, darkness, b..."


In [25]:
keywords.isna().sum()

id            0
keywords    215
dtype: int64

There are only 614 films without keywords

## Add the extra details to the films

Finally we will add the directors, actors and keywords to their corresponding film in the movies metadata csv file 

In [37]:
mdf = pd.read_parquet('../Data/Raw/movies_metadata.parquet')
# Select only the relevant columns
mdf = mdf[['movieId', 'id', 'title', 'genres', 'overview', 
           'runtime', 'release_date', 'tagline', 'vote_count', 
           'vote_average', 'poster_path', 'backdrop_path']]
# Select the filtered movies
mdf = mdf[mdf['id'].isin(ids['id'])]

In [38]:
# Merge the DataFrames
df = pd.merge(pd.merge(mdf, credits, on='id'), keywords, on='id')
df.head().transpose()

,0,1,2,3,4
movieId,10,1,6,2,11
id,710,862,949,8844,9087
title,GoldenEye,Toy Story,Heat,Jumanji,The American President
genres,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 16, 'name': 'Animation'}, {'id': 12, '...","[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam..."
overview,When a powerful satellite system falls into th...,"Led by Woody, Andy's toys live happily in his ...",Obsessive master thief Neil McCauley leads a t...,When siblings Judy and Peter discover an encha...,"Widowed U.S. president Andrew Shepherd, one of..."
runtime,130,81,170,104,113
release_date,1995-11-16,1995-11-22,1995-12-15,1995-12-15,1995-11-17
tagline,No limits. No fears. No substitutes.,The adventure takes off when toys come to life!,A Los Angeles crime saga.,Roll the dice and unleash the excitement!,Why can't the most powerful man in the world h...
vote_count,4080,19066,7756,10872,743
vote_average,6.894,7.969,7.923,7.24,6.534


We will also add the IMDb's score, which we previously saw, this can be helpful in the recommendation process.

In [39]:
m = df['vote_count'].min()
C = df['vote_average'].mean()

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

In [40]:
df['score'] = df.apply(weighted_rating, axis=1)

Finally to save later preprocessing I will extract the genres, since the are in JSON and set the null values of the tagline to an empty string.

In [43]:
# Extract genres
df['genres'] = df['genres'].apply(lambda val: [item['name'] for item in val])

# Delete the Nan in the tagline column
df['tagline'] = df['tagline'].fillna('')

See if we have duplicated values

In [44]:
rt = df['title'].duplicated().sum()
rtmdb = df['id'].duplicated().sum()
rid = df['movieId'].duplicated().sum()

print(f'There are {rt} duplicated titles, {rtmdb} duplicated TMDB ids, and {rid} duplicated movie ids.')

There are 54 duplicated titles, 0 duplicated TMDB ids, and 0 duplicated movie ids.


Delete the duplicated titles and select the one with the highest score

In [49]:
df_final = df.loc[df.groupby('title')['score'].idxmax()]
df_final.shape

(4946, 16)

In [50]:
rt = df_final['title'].duplicated().sum()
rtmdb = df_final['id'].duplicated().sum()
rid = df_final['movieId'].duplicated().sum()

print(f'There are {rt} duplicated titles, {rtmdb} duplicated TMDB ids, and {rid} duplicated movie ids.')

There are 0 duplicated titles, 0 duplicated TMDB ids, and 0 duplicated movie ids.


In [51]:
df_final['movieId'] = df_final['movieId'].astype(int)
df_final.head()

,movieId,id,title,genres,overview,runtime,release_date,tagline,vote_count,vote_average,poster_path,backdrop_path,cast,director,keywords,score
4169,221850,614696,#Alive,"[Action, Horror, Science Fiction]","As a grisly virus rampages a city, a lone man ...",98,2020-06-24,You must survive.,1927,7.222,/lZPvLUMYEPLTE2df1VW5FHTYC8N.jpg,/k2SY15W9QXH9qL8f4a4BbytV1BE.jpg,"[Yoo Ah-in, Park Shin-hye, Lee Hyun-wook]",Cho Il,"[escape, alone, survival, drone, zombie, apart...",7.220984
2353,117867,252178,'71,"[Thriller, Action, Drama, War]",A young British soldier must find his way back...,99,2014-10-10,,1155,6.802,/wbhqBocsP7QoX8SZLvCsGOWUAaQ.jpg,/aTloiKdNs2c8vlstbx3wBWD6Thi.jpg,"[Jack O'Connell, Sean Harris, Paul Anderson]",Yann Demange,"[1970s, riot, northern ireland, survival, sold...",6.822391
1463,69757,19913,(500) Days of Summer,"[Comedy, Drama, Romance]","Tom, greeting-card writer and hopeless romanti...",95,2009-07-17,This is not a love story. This is a story abou...,10548,7.296,/qXAuQ9hF30sQRsXf40OfRVl0MJZ.jpg,/1M2i4Mxd03elGOTmEkIvqrHfmyS.jpg,"[Joseph Gordon-Levitt, Zooey Deschanel, Geoffr...",Marc Webb,"[jealousy, gallery, fight, date, architect, in...",7.295363
2814,152077,333371,10 Cloverfield Lane,"[Thriller, Science Fiction, Drama, Horror]","After a catastrophic car crash, a young woman ...",104,2016-03-10,Monsters come in many forms.,8238,6.992,/84Dhwz93vCin6T1PX6ctSvWEuNE.jpg,/veGaHYcRHFPEoKfqxKbCEXI8tOT.jpg,"[John Goodman, Mary Elizabeth Winstead, John G...",Dan Trachtenberg,"[kidnapping, paranoia, bunker, basement, survi...",6.993529
271,2572,4951,10 Things I Hate About You,"[Comedy, Romance, Drama]","On the first day at his new school, Cameron in...",97,1999-03-30,How do I loathe thee? Let me count the ways.,8498,7.596,/ujERk3aKABXU3NDXOAxEQYTHe9A.jpg,/yvPbncYhMu9FfTjDhq0N5lgnVkO.jpg,"[Heath Ledger, Julia Stiles, Joseph Gordon-Lev...",Gil Junger,"[high school, deception, based on play or musi...",7.592968


## Save the Final Dataset

Once we have fetched the data, added it to the CSV metadata file, and removed any duplicate titles, we will save it as the final CSV file, which will be used for the recommender system

In [52]:
# Save it as the final dataset
df_final.to_csv('../data/processed/movies_final.csv', index=False)